# Chemosensing System Demonstration
This notebook demonstrates the end-to-end workflow of the image-based chemosensing system.

In [ ]:
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.append(os.path.abspath("../src"))

from chemosense.color_science import get_color_parameters
from chemosense.quantification import train_calibration_model, predict_concentration
from chemosense.analysis import run_pca

## 1. Simulated Data Generation
We'll simulate RGB values for a set of samples with increasing concentration.

In [ ]:
concentrations = np.array([0, 10, 20, 30, 40, 50])
# Simulate color deepening (decreasing intensity)
rgb_data = []
for c in concentrations:
    r = 200 - c * 2
    g = 150 - c * 1.5
    b = 100 - c * 1
    rgb_data.append([r, g, b])

rgb_array = np.array(rgb_data)
blank_rgb = rgb_array[0]

print("Simulated RGB Data:")
print(rgb_array)

## 2. Color Conversion and Delta E Calculation

In [ ]:
results = []
for i, rgb in enumerate(rgb_array):
    params = get_color_parameters(rgb, reference_rgb=blank_rgb)
    params['concentration'] = concentrations[i]
    results.append(params)

df = pd.DataFrame(results)
df

## 3. Calibration Curve

In [ ]:
model_data = train_calibration_model(df['delta_e'].values, df['concentration'].values)
print(f"Calibration Model Metrics: {model_data['metrics']}")

plt.figure(figsize=(8, 5))
plt.scatter(df['delta_e'], df['concentration'], label='Data Points')
plt.plot(df['delta_e'], model_data['model'].predict(df['delta_e'].values.reshape(-1, 1)), color='red', label='Linear Fit')
plt.xlabel('Delta E')
plt.ylabel('Concentration')
plt.title('Calibration Curve')
plt.legend()
plt.grid(True)
plt.show()